# Exercise 3: Variational Autoencoders
---
![VAE-cover](images/vae_cover.png)
---
## Table of Contents
1. [Introduction](#introduction)
2. [Setup and Configuration](#setup-and-configuration)
3. [Data Preparation](#data-preparation)
    - [The Dataset](#the-dataset)
    - [Data Loading](#data-loading)
4. [Model Architecture](#model-architecture)
    - [Encoder](#encoder-mapping-inputs-to-a-distribution)
    - [Decoder](#decoder-reconstructing-the-input)
    - [The Reparameterization Trick](#putting-it-together-the-reparameterization-trick)
5. [Training the Model](#training-the-model)
    - [Loss Function and Optimizer](#loss-function-and-optimizer)
    - [Training Loop](#training-loop)
6. [Result Visualization](#result-visualization)
7. [Exploring the Latent Space](#exploring-the-latent-space)

## Kaggle Environment Setup:

Here we will explain a basic way of setting up this notebook in Kaggle, such that you can work on the exercises in the online environment. This is for students who have had difficulties installing the environment/dependencies, or simply having issues initializing the kernel.

### Creating the Notebook

Creating the notebook can be done by simply pressing the create button in the top left corner of the kaggle home page. From there select import notebook. This will open a window which will allow you to navigate to your local repository where you should find `this notebook` (NOT the ex_3.ipynb notebook) and then open. This will then create the notebook in kaggle for you.

### Adding the Dataset

**You do not need to download or upload anything.** The data for this exercise is already published on Kaggle — you only have to attach it to your notebook.

1. In your notebook, find the `Input` section in the panel on the **right** and press **`+ Add Input`**.

2. In the window that opens, select the **`Datasets`** tab **first**. If you skip this you will be searching notebooks or models instead, and you will not find it.

3. Type `ae4353_3` into the search box.

4. Select the dataset uploaded by **Quentin Missinne**.

The dataset is now attached and appears in the file browser on the right, together with the `additional` folder of helper scripts. Then hover over the dataset folder, click the **copy-path** icon, and paste the result into `DATASET_PATH` in the configuration cell below.

### Enabling the GPU

This exercise trains a network on images, so it is worth switching the accelerator on. Open the `Settings` menu (or the three dots in the top right) → `Accelerator` → pick a **GPU**. No internet access is required for this exercise.

## Introduction
In this exercise, you will build and apply variational autoencoders to the [Flickr-Faces-HQ (FFHQ) dataset](https://github.com/NVlabs/ffhq-dataset), a large collection of photographs of human faces. While VAEs may no longer top the race in terms of generation quality compared to newer frameworks such as diffusion models, they remain a critical tool in various applications. VAEs are not only useful for data generation but also for tasks like anomaly detection, semi-supervised learning, and feature selection. In reinforcement learning, for instance, VAEs are employed to learn compact, informative representations of the environment, which can simplify state-space representations and improve policy learning. They also play a crucial role in disentangling latent features, aiding in interpretable and controllable generative processes.

Faces are a classic testbed for generative models, and for good reason: the things that differ between two portraits — pose, expression, lighting, hair, age — are exactly the kind of smooth, continuous factors of variation that a VAE is built to capture. By the end of this exercise you will be able to take two faces, encode them, and walk in a straight line from one to the other through the latent space, watching one person gradually turn into another.

> ℹ️ The faces come from [FFHQ](https://github.com/NVlabs/ffhq-dataset) (Karras et al., NVIDIA): photographs published on Flickr under Creative Commons or public-domain licenses, then aligned and cropped. We have cropped them further, converted them to grayscale and shrunk them to 32x32 so the exercise trains in minutes rather than hours. The collection is distributed under [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/), so please keep it to your coursework. NVIDIA also asks that FFHQ not be used to develop or improve facial recognition technology — building a generative model on it, as you are about to, is well outside that.

In [ ]:
# Import external modules

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import numpy as np

from tqdm import tqdm

## Setup and Configuration
---
Before building the model, we need to define some basic parameters and understand the shape of our data. The faces in this dataset are ***32x32 pixels***, and since they are grayscale, ***they only have a single channel***. Our encoder is a plain multi-layer perceptron, so each image arrives as one flat vector — which fixes the input dimension at 32 x 32 = ***1024***.

We also need to decide the size of the ***latent space*** - This is the dimensionality of the vector where the encoder compresses the input. This is considered the ***bottleneck*** of the VAE - The compressed representation of the data. ***Too small*** and the model cannot capture the variations in the data, reconstruction will lose details. ***Too large*** and the latent space may become under-regularized, leading to poor generative properties (e.g., sampling from the latent space produces meaningless images).

> 💡 Practical rule for faces: start somewhere around 20-64 dimensions. Faces vary along many more meaningful axes than handwritten digits do, so a latent space that is too tight will blur every face towards the same average one. Try a range of values (even if the first one comes out as expected) to see what different latent space dimensions do to a VAE.

You will also want to define other important parameters such as ***batch size***, ***learning rate***, ***hidden dimensions***, ***epochs*** and ***beta*** — the weight on the KL term. Beta is explained in [the loss section](#loss-function-and-optimizer); ***0.5*** is a good place to start, and it is the single most interesting number on this list to play with.

<strong style="color:red;">TODO 1.1: Set the hyperparameters for your model:</strong>

In [ ]:
# ---------------------------------------------------------------------------
#  Paste the path you copied from the Input panel on the right here:
# ---------------------------------------------------------------------------
DATASET_PATH = "/kaggle/input/ae4353-3"
# ---------------------------------------------------------------------------

dataset_file = f"{DATASET_PATH}/FFHQ_FACES.npz"

cuda = torch.cuda.is_available()
DEVICE = torch.device("cuda" if cuda else "cpu")


# ---------------------------------------------------------------------------
# TODO 1.1: Set hyperparameters
# ---------------------------------------------------------------------------

batch_size = ...

x_dim = ...
hidden_dim = ...
latent_dim = ...

lr = ...

epochs = ...

beta = ...

# ---------------------------------------------------------------------------
# END TODO 1.1
# ---------------------------------------------------------------------------

## Data Preparation
---
### The Dataset
FFHQ ships as 70,000 color photographs. Feeding those to a small MLP directly would be slow and would spend most of its capacity on background rather than faces, so we have prepared the data for you. Every image was:

1. **Center-cropped** to about 80% of its width — FFHQ aligns every face the same way, but leaves a generous margin of background around it.
2. **Converted to grayscale**, dropping the color channels.
3. **Downsampled to 32x32** pixels.

The result is `FFHQ_FACES.npz`, holding a `train` split of 60,000 faces and a `test` split of 10,000 — the same split sizes as MNIST, which this exercise used in previous years.

The `FaceDataset` class in `additional/dataset.py` reads that file and hands back each face as a flat vector of 1024 values scaled to `[0, 1]`. It returns a `(face, 0)` pair rather than a bare tensor: there are no labels here, and the `0` is just a placeholder that keeps the dataset unpacking the same way a labeled dataset would.

> ⚠️ We have provided the code for the dataset and the plotting utilities to save you time during the exercises. However, if you're unfamiliar with how to implement these modules, we encourage you to review the code. You will need to implement them on your own for the competition, and they may also be tested in the final exam.

In [ ]:
# Copy the helper scripts into the working directory so they can be imported

import shutil
import os

if not os.path.isdir(DATASET_PATH):
    raise FileNotFoundError(
        f"There is no folder at {DATASET_PATH}.\n"
        "Copy the real path from the Input panel on the right: expand the "
        "ae4353_3 dataset, hover over its folder, and click the copy-path icon."
    )

shutil.copytree(f"{DATASET_PATH}/additional", "/kaggle/working/additional", dirs_exist_ok=True)

Run the cell below to see what that preprocessing actually did. The top row is the original photograph, the bottom row is the 32x32 grayscale version your VAE will be trained on.

In [ ]:
from additional.plots import plot_preprocessing

plot_preprocessing(dataset_file, n=6)

### Data Loading
With the dataset defined, the `DataLoader` takes care of batching and shuffling. Two arguments are worth pointing out:

- **`shuffle=True`** on the training loader reshuffles the faces every epoch, so the model never learns anything from the order they arrive in.
- **`drop_last=True`** throws away the final, smaller batch of each epoch, so every batch holds exactly `batch_size` faces. Nothing forces you to do this, but 60,000 does not divide evenly by most batch sizes, and it saves you a confusing shape error later if you reshape with a hard-coded `batch_size` in your training loop.

Please execute the cell below to build the datasets and their loaders.

In [ ]:
from additional.dataset import FaceDataset

train_dataset = FaceDataset(dataset_file, split="train")
test_dataset = FaceDataset(dataset_file, split="test")

kwargs = {"num_workers": 1, "pin_memory": True} if cuda else {}

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, drop_last=True, **kwargs
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, drop_last=True, **kwargs
)

print(f"Training on {len(train_dataset)} faces, testing on {len(test_dataset)}.")
print(f"Running on {DEVICE}.")

## Model Architecture
---
![VAE Diagram](./images/vae_diagram.jpg)

(this image comes from [pyimagesearch](https://pyimagesearch.com/2023/10/02/a-deep-dive-into-variational-autoencoders-with-pytorch/). A full explanation of how a VAE works in depth can be found there too!)

### Encoder: Mapping inputs to a distribution

The encoder's role is to compress the input image into a representation in the latent space. But unlike a standard autoencoder, the VAE encoder outputs a **distribution**, not a single vector. For each input ***x***, the encoder produces two vectors:
- **mean vector** ($\mu$) - the center of the latent distribution
- **Log-variance vector** (log $\sigma^2$) - describes how spread out the distribution is.

Formally:

$(\mu, log(\sigma^2))$ = Encoder($x$)

This probabilistic encoding allows the latent space to capture both **what features are important** and **how uncertain the model is** about them.

<strong style="color:red;">TODO 2.1: Define the Encoder:</strong>

In [ ]:
# ----------------------------------------------------------------------------
# TODO 2.1: Implement the Encoder
# ----------------------------------------------------------------------------

class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(Encoder, self).__init__()
        
        pass
        

    def forward(self, x):
        pass
        return z_mean, z_log_var

# ----------------------------------------------------------------------------
# END TODO 2.1
# ----------------------------------------------------------------------------

### Decoder: Reconstructing the input

The decoder takes a point from the latent space and attempts to reconstruct the original image.

$\hat x$ = Decoder($z$)

Here $\hat x$ is the reconstructed image, and its similarity to the original input $x$ is measured using the **reconstruction loss**. If the encoder has captured the right features, the decoder can recreate images that look very much like the originals.

<strong style="color:red;">TODO 2.2: Define the Decoder:</strong>

In [ ]:
# ----------------------------------------------------------------------------
# TODO 2.2: Implement the Decoder
# ----------------------------------------------------------------------------

class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super(Decoder, self).__init__()

        pass

    def forward(self, x):
        pass
        return x_hat

# ----------------------------------------------------------------------------
# END TODO 2.2
# ----------------------------------------------------------------------------

### Putting it together: the reparameterization trick

To connect the encoder and decoder, we need to sample a latent vector $z$ from the distribution produced by the encoder. However, naive sampling would break backpropagation. The solution is the **reparameterization trick**:

$z = \mu + \sigma \odot \epsilon$

$\epsilon \sim \mathcal{N}(0, I)$

where: 
- $\mu$ is the mean from the encoder
- $\sigma$ = $exp(0.5 * log (\sigma^2))$ is the standard deviation
- $\epsilon$ is random noise drawn from a standard normal distribution
- $\odot$ denotes element-wise multiplication.

This formulation keeps the randomness while allowing gradients to flow, making the model trainable. By combining **probabilistic encoding**, **differentiable sampling**, and **decoding**, the VAE learns a **smooth and continuous latent space**. This not only enables faithful reconstruction of faces but also makes it possible to generate **entirely new faces** by drawing random vectors from the latent space.

<strong style="color:red;">TODO 2.3: Implement the full VAE:</strong>

In [ ]:
# ----------------------------------------------------------------------------
# TODO 2.3: Implement the VAE
# ----------------------------------------------------------------------------

class VAE(nn.Module):
    def __init__(self, Encoder, Decoder):
        super(VAE, self).__init__()
        
        # Define the encoder and decoder
        pass

    def reparameterization(self, mean, var):
        pass
        return z

    def forward(self, x):
        pass
        return x_hat, mean, log_var

# ----------------------------------------------------------------------------
# END TODO 2.3
# ----------------------------------------------------------------------------

In [ ]:
encoder = Encoder(input_dim=x_dim, hidden_dim=hidden_dim, latent_dim=latent_dim)
decoder = Decoder(latent_dim=latent_dim, hidden_dim=hidden_dim, output_dim=x_dim)

model = VAE(Encoder=encoder, Decoder=decoder).to(DEVICE)

## Training the Model
---
### Loss Function and Optimizer

Now that the model is defined, we need two more ingredients before we can start training: 
1. A **loss function** that tells the model how well it is doing.
2. An **optimizer** that updates the model's parameters based on that loss.

---

### The VAE loss function

The loss in the VAE has two parts: 

1. **Reconstruction loss**:
    - Measures how close the reconstructed image $\hat x$ is to the original input image $x$.
    - Here, we use **binary cross-entropy (BCE)**, summed over all pixels.
    - Essentially the model gets penalized if it can't recreate the input faces correctly.

    $\mathrm{recon} = \mathrm{BCE}(x, \hat x)$

    > 💡 Our pixels are grayscale values in $[0, 1]$ rather than hard 0s and 1s. BCE is still the standard choice here — it simply measures the disagreement between two values in $[0, 1]$, and it penalizes confident mistakes much harder than a plain squared error would.

2. **KL divergence loss**:
    - Regularizes the latent space so that the encoded distribution $\mathcal{N}(\mu, \sigma^2)$ stays close to a standard normal distribution $\mathcal{N}(0, I)$. 
    - This keeps the latent space smooth and ensures that sampling random points produces meaningful faces.

    $\mathrm{KL} = - 0.5 \sum(1 + log(\sigma^2) - \mu^2 - \sigma^2)$
    
    The final loss is the sum of both terms (with an optional scaling factor $\beta$ to control the strength of the KL term):

    $\mathcal{L} = \mathrm{recon} + \beta * KL$

    Dividing by the batch size helps keep the loss values stable.

> 💡 **$\beta$ is the most interesting knob in this exercise.** Setting $\beta = 1$ optimizes the exact ELBO, which is the principled choice — but on this dataset it causes **posterior collapse**. Zeroing out a latent dimension is a cheap way to reduce the KL term, so the model quietly stops using most of them: at $\beta = 1$ only about **a third** of them survive. The decoder is then left with too few numbers to describe a face, and every reconstruction drifts towards the same average face.
>
> Start at $\beta = 0.5$, which keeps substantially more of them alive. Once it works, move it in both directions and watch what happens: push it back towards 1 and your reconstructions blur towards the mean; drop it to 0.1 and they sharpen, but the latent space drifts further from the $\mathcal{N}(0, I)$ you will be sampling from in the last section. That tension between **reconstruction fidelity** and **latent-space regularity** is the entire balancing act of a VAE, and $\beta$ is how you pick a point on it.

### The optimizer:

To train the model we use the **Adam optimizer**. Adam adapts the learning rate for each parameter, making training faster and more stable than just standard gradient descent. Therefore to make it run we pass in the `model parameters` and the `learning rate`.

<strong style="color:red;">TODO 3.1-3.2: Define the loss function and the optimizer:</strong>

In [ ]:
from torch.optim import Adam
import torch.nn.functional as F

# ----------------------------------------------------------------------------
# TODO 3.1: Implement the loss function
# ----------------------------------------------------------------------------

# reconstruction + KL divergence losses summed over all elements and batch
def loss_function(x, x_hat, mean, log_var, beta=1.0):
    # reconstruction loss (BCE summed over pixels)
    pass

    # KL divergence term
    pass

    # normalize by batch size for stability
    return (recon + beta * kl) / x.size(0)

# ----------------------------------------------------------------------------
# END TODO 3.1
# ----------------------------------------------------------------------------

# ----------------------------------------------------------------------------
# TODO 3.2: Set up the optimizer
# ----------------------------------------------------------------------------

optimizer = ...

# ----------------------------------------------------------------------------
# END TODO 3.2
# ----------------------------------------------------------------------------

### Training Loop

With the model, loss, and optimizer defined, we can now put everything together in a training loop. Training a VAE looks similar to training other neural networks, but we track three key values:  

1. **Reconstruction loss** - measures how well the decoder reproduces the input images.  
2. **KL divergence** - regularizes the latent space so it stays close to a standard normal distribution.  
3. **Total loss** - the sum of the two, which the optimizer minimizes.  

At each epoch:  
- We loop through the training batches and pass the images through the encoder and decoder.  
- We compute the reconstruction and KL terms separately to better monitor how the model is learning.  
- The gradients are reset (`optimizer.zero_grad()`), the loss is backpropagated (`loss.backward()`), and the optimizer updates the model's parameters (`optimizer.step()`).  

Using `tqdm`, we also show a progress bar with the current loss values for each batch, making it easier to see improvements during training. At the end of each epoch, we print the average losses so we can track the overall learning progress.  

By monitoring both reconstruction and KL divergence, we ensure the model is **balancing accurate reconstructions with a well-structured latent space**, which is the essence of training a VAE.  

<strong style="color:red;">TODO 4.1: Write and run the training loop:</strong>

In [ ]:
print("Start training VAE...")

# ----------------------------------------------------------------------------
# TODO 4.1: Implement the training loop
# ----------------------------------------------------------------------------

model.train()

for epoch in range(epochs):
    pass

# ----------------------------------------------------------------------------
# END TODO 4.1
# ----------------------------------------------------------------------------

print("Finish!!")

## Result Visualization
---
We provide a simple overview of your results. The top row shows the input image and the bottom row shows the images the network reconstructs. Feel free to create other meaningful visualizations, as it tests your ability to check the performance of your models!

> 💡 Do not expect crisp photographs. A VAE trained with this loss produces noticeably smooth, slightly blurry faces — that is the well-known cost of averaging over everything the latent code cannot express, not a bug in your implementation.

> 🚀 **Want to push further? Change the architecture — you are completely free to.** Nothing in this notebook depends on the two-hidden-layer MLP we suggested; the rest of the code only cares that `Encoder` returns `(mean, log_var)` and `Decoder` returns an image. The cheapest experiment is simply going deeper: add a third or fourth `nn.Linear` layer to each, or widen `hidden_dim`. Beyond that, a **convolutional** encoder and decoder (`nn.Conv2d` going down, `nn.ConvTranspose2d` coming back up) is the change that really pays off on images, because it lets the model exploit the fact that neighboring pixels are related instead of treating all 1024 of them as an unordered list. If you go that route, ask the dataset for images rather than flat vectors:
>
> ```python
> train_dataset = FaceDataset(dataset_file, split="train", flatten=False)   # (1, 32, 32) instead of (1024,)
> ```
>
> You will then need to drop the `x.view(x.size(0), x_dim)` reshape in the training loop, and pass `img_shape` through to the plotting helpers. Try things — comparing a deeper model against your first one, and explaining *why* it does or does not help, is worth more than any single number.

In [ ]:
from additional.plots import plot_reconstructions

plot_reconstructions(model, test_loader, DEVICE)

## Exploring the Latent Space
---
Reconstruction only shows that the model can copy an image it was given. The more interesting question is what lives *between* the training images.

Because the KL term pushed every encoded face towards the same standard normal distribution, the latent space should have no holes in it: every point you pick should decode to something face-like. Two experiments test that.

**First, generation.** Sample a latent vector straight from $\mathcal{N}(0, I)$ — no input image at all — and decode it. Run the cell below.

In [ ]:
from additional.plots import plot_latent_samples

plot_latent_samples(model, latent_dim, DEVICE)

**Second, interpolation.** Take two real faces, encode both, and walk in a straight line from one latent vector to the other, decoding at each step. If the latent space really is smooth, you will see one face morph continuously into the other, with every intermediate frame still looking like a plausible person.

You need to:
1. Encode both faces and keep the **mean** of each latent distribution (we want a deterministic walk, so ignore the sampling step here).
2. Build `n_steps` latent vectors spaced evenly along the line from `z_a` to `z_b`.
3. Decode them into images.

<strong style="color:red;">TODO 5.1: Interpolate between two faces:</strong>

In [ ]:
from additional.plots import plot_interpolation

model.eval()

x, _ = next(iter(test_loader))
x = x.to(DEVICE)

# Pick the two most dissimilar faces in the batch, so the morph is easy to see.
# Feel free to replace these with indices of your own choosing.
i, j = divmod(int(torch.cdist(x, x).argmax()), x.size(0))
print(f"Interpolating between faces {i} and {j} of the batch.")

face_a = x[i].unsqueeze(0)
face_b = x[j].unsqueeze(0)

n_steps = 10

# ----------------------------------------------------------------------------
# TODO 5.1: Interpolate between two faces
# ----------------------------------------------------------------------------

with torch.no_grad():
    # Encode both faces, keeping only the mean of each latent distribution
    mean_a = ...
    mean_b = ...

    # Build the latent vectors along the path, then decode them
    z_path = ...
    faces = ...

# ----------------------------------------------------------------------------
# END TODO 5.1
# ----------------------------------------------------------------------------

plot_interpolation(faces)

> 🤔 If the walk jumps abruptly somewhere in the middle, or passes through frames that look like nothing at all, that is a sign the latent space has holes in it — usually because the KL term is too weak relative to the reconstruction term. Try raising $\beta$ and training again, and watch what it costs you in reconstruction sharpness.

## Solutions:
---
Below is one correct implementation of each `TODO`, for you to compare against once you have had a proper go yourself.

> ⚠️ **The hyperparameters in the first solution cell are not tuned.** They are chosen to demonstrate a working implementation, not a good model. Apart from `x_dim`, which is fixed by the 32x32 image size, every value is open — and getting a feel for how `latent_dim`, `hidden_dim`, `beta` and `epochs` change the reconstructions and the latent space is the more interesting half of this exercise.

In [ ]:
# ---------------------------------------------------------------------------
#  Paste the path you copied from the Input panel on the right here:
# ---------------------------------------------------------------------------
DATASET_PATH = "/kaggle/input/ae4353-3"
# ---------------------------------------------------------------------------

dataset_file = f"{DATASET_PATH}/FFHQ_FACES.npz"

cuda = torch.cuda.is_available()
DEVICE = torch.device("cuda" if cuda else "cpu")


# ---------------------------------------------------------------------------
# TODO 1.1: Set hyperparameters
# ---------------------------------------------------------------------------
# NOTE: these values are NOT tuned. They are here to show a correctly shaped,
# working implementation -- nothing more. Only `x_dim` is forced on you (it has
# to match the 32x32 images); every other number below is a plausible starting
# point that we picked, not an answer. Finding good values is part of the
# exercise, so treat these as the beginning of your search rather than the end.

batch_size = 128

x_dim = 1024
hidden_dim = 512
latent_dim = 32

lr = 1e-3

epochs = 50

beta = 0.5   # see the loss section -- beta = 1 collapses the latent space here

# ---------------------------------------------------------------------------
# END TODO 1.1
# ---------------------------------------------------------------------------

In [ ]:
# LOADING MODEL:

"""
simple Gaussian MLP Encoder and Decoder
"""
# ----------------------------------------------------------------------------
# TODO 2.1: Implement the Encoder
# ----------------------------------------------------------------------------

class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(Encoder, self).__init__()

        self.fc_input = nn.Linear(input_dim, hidden_dim)
        self.fc_input2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc2_mean = nn.Linear(hidden_dim, latent_dim)
        self.fc2_log_var = nn.Linear(hidden_dim, latent_dim)

        self.ReLU = nn.ReLU()

        self.training = True

    def forward(self, x):
        h = self.ReLU(self.fc_input(x))
        h = self.ReLU(self.fc_input2(h))
        z_mean = self.fc2_mean(h)
        z_log_var = self.fc2_log_var(h)
        return z_mean, z_log_var

# ----------------------------------------------------------------------------
# END TODO 2.1
# ----------------------------------------------------------------------------

In [ ]:
# ----------------------------------------------------------------------------
# TODO 2.2: Implement the Decoder
# ----------------------------------------------------------------------------

class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super(Decoder, self).__init__()

        self.fc_hidden = nn.Linear(latent_dim, hidden_dim)
        self.fc_hidden2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc_output = nn.Linear(hidden_dim, output_dim)

        self.ReLU = nn.ReLU()

        self.training = True

    def forward(self, x):
        h = self.ReLU(self.fc_hidden(x))
        h = self.ReLU(self.fc_hidden2(h))

        x_hat = torch.sigmoid(self.fc_output(h))
        return x_hat

# ----------------------------------------------------------------------------
# END TODO 2.2
# ----------------------------------------------------------------------------

In [ ]:
# ----------------------------------------------------------------------------
# TODO 2.3: Implement the VAE
# ----------------------------------------------------------------------------

class VAE(nn.Module):
    def __init__(self, Encoder, Decoder):
        super(VAE, self).__init__()
        self.Encoder = Encoder
        self.Decoder = Decoder

    def reparameterization(self, mean, var):
        epsilon = torch.randn_like(var).to(DEVICE)
        z = mean + var * epsilon
        return z

    def forward(self, x):
        mean, log_var = self.Encoder(x)
        z = self.reparameterization(mean, torch.exp(0.5 * log_var))
        x_hat = self.Decoder(z)

        return x_hat, mean, log_var

# ----------------------------------------------------------------------------
# END TODO 2.3
# ----------------------------------------------------------------------------

In [ ]:
from torch.optim import Adam
import torch.nn.functional as F

# ----------------------------------------------------------------------------
# TODO 3.1: Implement the loss function
# ----------------------------------------------------------------------------

# reconstruction + KL divergence losses summed over all elements and batch
def loss_function(x, x_hat, mean, log_var, beta=1.0):
    # reconstruction loss (BCE summed over pixels)
    recon = F.binary_cross_entropy(x_hat, x, reduction="sum")

    # KL divergence term
    kl = -0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp())

    # normalize by batch size for stability
    return (recon + beta * kl) / x.size(0)

# ----------------------------------------------------------------------------
# END TODO 3.1
# ----------------------------------------------------------------------------

# ----------------------------------------------------------------------------
# TODO 3.2: Set up the optimizer
# ----------------------------------------------------------------------------

optimizer = Adam(model.parameters(), lr=lr)

# ----------------------------------------------------------------------------
# END TODO 3.2
# ----------------------------------------------------------------------------

In [ ]:
print("Start training VAE...")

# ----------------------------------------------------------------------------
# TODO 4.1: Implement the training loop
# ----------------------------------------------------------------------------

model.train()

for epoch in range(epochs):
    overall_loss = 0
    overall_recon = 0
    overall_kl = 0

    loop = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}")

    for batch_idx, (x, _) in loop:
        x = x.view(x.size(0), x_dim).to(DEVICE)
        optimizer.zero_grad()

        x_hat, mean, log_var = model(x)

        # Compute separate components
        recon = F.binary_cross_entropy(x_hat, x, reduction="sum") / x.size(0)
        kl = (-0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp())) / x.size(0)
        loss = recon + beta * kl

        loss.backward()
        optimizer.step()

        overall_loss += loss.item()
        overall_recon += recon.item()
        overall_kl += kl.item()

        # Update tqdm with current batch components
        loop.set_postfix(loss=loss.item(), recon=recon.item(), kl=kl.item())

    avg_loss = overall_loss / len(train_loader)
    avg_recon = overall_recon / len(train_loader)
    avg_kl = overall_kl / len(train_loader)

    print(f"\tEpoch {epoch + 1} complete! "
          f"Average Loss: {avg_loss:.4f}, Recon: {avg_recon:.4f}, KL: {avg_kl:.4f}")

# ----------------------------------------------------------------------------
# END TODO 4.1
# ----------------------------------------------------------------------------

print("Finish!!")

In [ ]:
from additional.plots import plot_interpolation

# ----------------------------------------------------------------------------
# TODO 5.1: Interpolate between two faces
# ----------------------------------------------------------------------------

with torch.no_grad():
    # Encode both faces, keeping only the mean of each latent distribution
    mean_a, _ = model.Encoder(face_a)
    mean_b, _ = model.Encoder(face_b)

    # Build the latent vectors along the path, then decode them
    alphas = torch.linspace(0, 1, n_steps, device=DEVICE).unsqueeze(1)
    z_path = (1 - alphas) * mean_a + alphas * mean_b
    faces = model.Decoder(z_path)

# ----------------------------------------------------------------------------
# END TODO 5.1
# ----------------------------------------------------------------------------

plot_interpolation(faces)